# Aula 2 - Testes Automatizados para Modelos de IA
## TDD aplicado a ML + Testes baseados em propriedades e invariantes

IEC PUC Minas - Engenharia de Inteligência Artificial e MLOps

## Setup

Além do `ipytest` (Aula 1), hoje usamos também o [`hypothesis`](https://hypothesis.readthedocs.io/), biblioteca de testes baseados em propriedades.

In [ ]:
!pip install -q "ipytest==0.14.*" "hypothesis==6.164.*"

import ipytest
import pytest
ipytest.autoconfig()

---
## 1. TDD passo a passo - construindo `clip_score`

Vamos construir, ao vivo, uma função que limita um score entre 0 e 1 (comum em pipelines de ML pra normalizar saídas).

### 🔴 Passo 1 - Red

Escrevemos o teste antes da função existir. Deve falhar (`clip_score` nem existe ainda) - isso é o ponto de partida esperado do TDD, não um erro nosso.

In [ ]:
%%ipytest

# garante o estado inicial do TDD mesmo se você já rodou células mais à frente
try:
    del clip_score
except NameError:
    pass


def test_clip_score_clips_above_max():
    assert clip_score(1.5) == 1.0


### 🟢 Passo 2 - Green (o mínimo possível)

Implementação propositalmente incompleta ('fake it') - só o suficiente pra passar este teste específico.

In [ ]:
%%ipytest

def clip_score(x):
    return 1.0


def test_clip_score_clips_above_max():
    assert clip_score(1.5) == 1.0


### 🔴🟢 Passo 3 - Um novo teste força a implementação real

Com a implementação 'fake', o primeiro teste continua passando, mas o novo falha - a implementação incompleta não sobrevive a um segundo caso.

In [ ]:
%%ipytest

def clip_score(x):
    return 1.0


def test_clip_score_clips_above_max():
    assert clip_score(1.5) == 1.0


def test_clip_score_clips_below_min():
    assert clip_score(-0.5) == 0.0


### 🟢 Implementação real

Agora sim, a lógica completa - e um terceiro teste garantindo que valores dentro do intervalo não são alterados.

In [ ]:
%%ipytest

def clip_score(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))


def test_clip_score_clips_above_max():
    assert clip_score(1.5) == 1.0


def test_clip_score_clips_below_min():
    assert clip_score(-0.5) == 0.0


def test_clip_score_passes_through_value_in_range():
    assert clip_score(0.5) == 0.5


### 🔵 Refactor

Só refatoramos com os testes verdes. Aqui, um refactor bobo de propósito - renomear `lo`/`hi` para nomes mais descritivos - mostrando que os testes continuam passando.

In [ ]:
%%ipytest

def clip_score(x, min_val=0.0, max_val=1.0):
    """Limita x ao intervalo [min_val, max_val]."""
    return max(min_val, min(max_val, x))


def test_clip_score_clips_above_max():
    assert clip_score(1.5) == 1.0


def test_clip_score_clips_below_min():
    assert clip_score(-0.5) == 0.0


def test_clip_score_passes_through_value_in_range():
    assert clip_score(0.5) == 0.5


---
## 2. [Prática 1] TDD para `is_valid_score`

Agora é com vocês. `is_valid_score(x)` parece simples, mas tem decisões de design escondidas - tomem cada uma e defendam com um teste, seguindo o ciclo TDD (red → green → refactor), um teste por vez:

- `is_valid_score(float('nan'))` deve ser `False` - um dado inválido não pode passar disfarçado de válido.
- `is_valid_score(float('inf'))` - **decisão de vocês**: válido ou não? Comentem o porquê no código.
- `is_valid_score(True)` - cuidado: em Python `bool` é subtipo de `int` (`isinstance(True, int)` é `True`). Deveria contar como score válido?
- `is_valid_score("0.5")` - uma string numérica deveria ser aceita?
- `is_valid_score(3)` - um `int` normal deveria ser aceito (não só `float`)?

Não existe "gabarito único" pra `inf`, `bool` ou string numérica - o que importa é que a decisão fique **documentada em comentário** e **coberta por teste**.

In [ ]:
%%ipytest

import pytest
import math

# Decisões de design (documentadas e cobertas por teste):
# - inf: INVÁLIDO. Infinito costuma indicar overflow/erro numérico, não um
#   score de confiança utilizável em pipeline de ML.
# - bool: INVÁLIDO. Em Python, bool é subtipo de int (isinstance(True, int)
#   é True). Aceitar True/False disfarçaria um bug de tipo como score válido.
# - string numérica ('0.5'): INVÁLIDA. Validar não é converter — se a entrada
#   chegou como str, algo falhou antes e não devemos "consertar" silenciosamente.
# - int e float finitos (não-NaN): VÁLIDOS.


def is_valid_score(x) -> bool:
    """Retorna True se x for um score numérico finito (int ou float, não bool)."""
    if isinstance(x, bool):
        return False
    if isinstance(x, (int, float)):
        return math.isfinite(x)
    return False


def test_is_valid_score_rejects_nan():
    assert is_valid_score(float("nan")) is False


def test_is_valid_score_accepts_valid_number():
    assert is_valid_score(0.7) is True


def test_is_valid_score_rejects_non_numeric_string():
    assert is_valid_score("abc") is False


def test_is_valid_score_handles_infinity():
    # Decisão: inf é inválido (overflow/erro numérico, não score utilizável).
    assert is_valid_score(float("inf")) is False
    assert is_valid_score(float("-inf")) is False


def test_is_valid_score_handles_bool():
    # Decisão: bool não conta como score (evita True passar como int 1).
    assert is_valid_score(True) is False
    assert is_valid_score(False) is False


def test_is_valid_score_rejects_numeric_string():
    # Decisão: validar ≠ converter; string numérica é rejeitada.
    assert is_valid_score("0.5") is False


def test_is_valid_score_accepts_plain_int():
    assert is_valid_score(3) is True


---
## 3. Testes baseados em propriedades

Em vez de "para esta entrada, este é o resultado", testamos "para qualquer entrada válida, esta propriedade é sempre verdadeira". O `hypothesis` gera dezenas de entradas sozinho e procura um contraexemplo.

In [ ]:
%%ipytest --hypothesis-show-statistics

from hypothesis import given, strategies as st


def clip_score(x, min_val=0.0, max_val=1.0):
    return max(min_val, min(max_val, x))


@given(st.floats(allow_nan=False, allow_infinity=False))
def test_clip_score_always_within_bounds(x):
    result = clip_score(x)
    assert 0.0 <= result <= 1.0


In [ ]:
%%ipytest

from hypothesis import given, strategies as st


def clip_score(x, min_val=0.0, max_val=1.0):
    return max(min_val, min(max_val, x))


@given(st.floats(allow_nan=False, allow_infinity=False))
def test_clip_score_is_idempotent(x):
    once = clip_score(x)
    twice = clip_score(clip_score(x))
    assert once == twice


### Quando a propriedade pega um bug real

Simulando uma versão com um bug (esqueceu de aplicar o teto `max_val`). Rode e observe o `hypothesis` encontrar um contraexemplo e reduzi-lo (*shrinking*) ao caso mais simples que ainda quebra a propriedade - não é o menor valor matematicamente possível, é o mais simples de entender.

In [ ]:
%%ipytest

from hypothesis import given, strategies as st


def clip_score_buggy(x, min_val=0.0, max_val=1.0):

    return max(min_val, x)


@given(st.floats(allow_nan=False, allow_infinity=False))
def test_clip_score_buggy_within_bounds(x):
    result = clip_score_buggy(x)
    assert 0.0 <= result <= 1.0


### Um bug mais sutil: a propriedade também tem limites

Lembra da pergunta lá no início - "e se o valor de entrada for `float('nan')`?" O teste de propriedade que escrevemos usa `allow_nan=False`, então o `hypothesis` nunca gera `NaN` pra testar. Vamos ver o que `clip_score` (a versão **correta**, sem bug nenhum) faz com um `NaN` genuíno.

In [ ]:
def clip_score(x, min_val=0.0, max_val=1.0):
    return max(min_val, min(max_val, x))


nan = float("nan")
resultado = clip_score(nan)
print(f"clip_score(NaN) = {resultado}")
print("Um score inválido virou silenciosamente o score MÁXIMO de confiança.")
print("A propriedade que escrevemos não pega isso: ela nunca testou NaN")
print("(allow_nan=False), porque não faz parte do domínio que ela descreve.")
print()
print("Lição: teste de propriedade só verifica o que a propriedade diz.")
print("Validar a entrada (é isso que o exercício 'is_valid_score' faz)")
print("é uma etapa ANTES do clip, não depois.")

---
## 4. [Prática 2] TDD + propriedade para 
ormalize_batch

clip_score normaliza **um** valor. Pipelines em produção normalizam **lotes inteiros**. Implemente:

`
normalize_batch(scores: list[float]) -> list[float]
`

usando normalização min-max: (x - min(scores)) / (max(scores) - min(scores)).

### 🗺️ Roteiro desta prática (siga nesta ordem)

| Passo | O quê | Status |
|-------|--------|--------|
| **1️⃣** | Teste exemplo: [0, 5, 10] → [0, 0.5, 1] + fórmula min-max | ✅ feito |
| **2️⃣** | Lista constante (max == min): decidir o que fazer (não dividir por zero!) e testar | ⏳ próximo |
| **3️⃣** | 2 propriedades com Hypothesis: saída em [0,1] + ordem relativa preservada | ⬜ depois |

**Decisão de design pra vocês tomarem e testarem:** o que fazer quando max(scores) == min(scores) (todos os valores do lote são iguais)? Dividir por zero não é opção.

Depois de implementar com TDD, escrevam **2 testes de propriedade** com hypothesis:
1. Todo valor de saída está sempre em [0, 1].
2. A ordem relativa dos valores é preservada - se scores[i] <= scores[j] na entrada, o mesmo vale para os valores normalizados correspondentes.

Para a propriedade 2, pensem no caso de lote constante antes de escrever o assert.

> ⚠️ **Nota de produção (leia, não vamos discutir agora):** repare que 
ormalize_batch calcula min e max **a partir do próprio lote recebido**. Isso é ótimo pro exercício e é um **antipadrão em produção**: o mesmo score sai normalizado diferente dependendo de quem veio junto no lote, e o modelo em inferência passa a ver uma escala que ele nunca viu no treino - é o que se chama *train/serve skew*. Em pipeline real, min/max (ou média/desvio) são **congelados no treino** e aplicados como constantes fixas na hora de servir. O mesmo vale para a média e o desvio-padrão de lag_outliers, na Prática 5. Voltamos nisso quando falarmos de validação de dados.


In [ ]:
%%ipytest

import pytest
from hypothesis import given, strategies as st

# =============================================================================
# 🗺️ PRÁTICA 2 — normalize_batch
#   1️⃣ ✅ feito   — caso feliz [0, 5, 10] → [0, 0.5, 1]
#   2️⃣ ✅ feito   — lista constante → todos 0.0 (sem variação = base da escala)
#   3️⃣ parcialmente
#        3.1  ✅  "Todo valor de saída está sempre em [0, 1]"
#        3.2  ⬜  "Ordem relativa preservada"  ← próximo
# =============================================================================

# Decisão de design (passo 2): se max == min, retornar [0.0] * len(scores).
# Motivo: sem variação, não há escala; colocamos todos na base (0).


# --- FUNÇÃO ------------------------------------------------------------------
def normalize_batch(scores: list[float]) -> list[float]:
    """Normalização min-max: (x - min) / (max - min)."""
    lo, hi = min(scores), max(scores)
    span = hi - lo
    if span == 0:
        return [0.0] * len(scores)
    return [(x - lo) / span for x in scores]


# --- 1️⃣ TESTE: caso feliz ---------------------------------------------------
def test_normalize_batch_maps_min_to_zero_and_max_to_one():
    assert normalize_batch([0.0, 5.0, 10.0]) == [0.0, 0.5, 1.0]


# --- 2️⃣ TESTE: lista constante ----------------------------------------------
def test_normalize_batch_handles_constant_list():
    # Decisão: lote constante → todos 0.0
    assert normalize_batch([3.0, 3.0, 3.0]) == [0.0, 0.0, 0.0]


# --- 3️⃣ PROPRIEDADES Hypothesis ---------------------------------------------

# >>> 3️⃣.1 ✅ "Todo valor de saída está sempre em [0, 1]" <<<
@given(st.lists(st.floats(allow_nan=False, allow_infinity=False, width=32), min_size=1))
def test_normalize_batch_output_always_within_bounds(scores):
    result = normalize_batch(scores)
    assert all(0.0 <= x <= 1.0 for x in result)


# >>> 3️⃣.2 ⬜ "Ordem relativa preservada"  ← ainda NÃO feito <<<
# @given(st.lists(st.floats(allow_nan=False, allow_infinity=False, width=32), min_size=2))
# def test_normalize_batch_preserves_relative_order(scores):
#     pytest.fail("TODO passo 3.2: se scores[i] <= scores[j], normalizado[i] <= normalizado[j]")


---
## 5. [Prática 3] `clean_and_dedupe`

O `preprocess` da Aula 1 (`strip().lower()`) está pronto pra virar parte de um pipeline de limpeza de dataset real?

Implemente:
```
clean_and_dedupe(texts: list[str]) -> list[str]
```
que aplica `preprocess` a cada texto da lista e remove duplicatas, **preservando a ordem da primeira ocorrência**.

**Decisões de design pra documentar:** uma duplicata é decidida antes ou depois do `preprocess`? Uma string que vira `""` depois do `strip()` deveria ser mantida na saída ou descartada?

Depois de implementar com TDD, escrevam **3 testes de propriedade**:
1. **Idempotência do pipeline completo**: `clean_and_dedupe(clean_and_dedupe(xs)) == clean_and_dedupe(xs)`
2. **Sem duplicatas na saída**: `len(saida) == len(set(saida))`
3. **Ordem preservada**: a ordem relativa das primeiras ocorrências é mantida

In [ ]:
%%ipytest

import pytest
import string
from hypothesis import given, strategies as st


def preprocess(text: str) -> str:
    return text.strip().lower()


# TODO: implemente clean_and_dedupe(texts) seguindo TDD.
# Documente aqui suas decisões de design (duplicata antes/depois do preprocess,
# string vazia mantida ou descartada):


def test_clean_and_dedupe_applies_preprocess():
    pytest.fail("TODO: clean_and_dedupe(['  Foo  ']) -> ['foo']")


def test_clean_and_dedupe_removes_duplicates_preserving_first_occurrence():
    pytest.fail("TODO: clean_and_dedupe(['Foo', 'BAR', 'foo']) -> ['foo', 'bar']")


# TODO: descomente e complete as três propriedades abaixo

# @given(st.lists(st.text(alphabet=string.printable, max_size=10)))
# def test_clean_and_dedupe_is_idempotent(texts):
#     pytest.fail("TODO: aplicar duas vezes == aplicar uma vez")


# @given(st.lists(st.text(alphabet=string.printable, max_size=10)))
# def test_clean_and_dedupe_has_no_duplicates(texts):
#     pytest.fail("TODO: len(saida) == len(set(saida))")


# @given(st.lists(st.text(alphabet=string.printable, max_size=10)))
# def test_clean_and_dedupe_preserves_order(texts):
#     pytest.fail("TODO: comparar saida com a lista de primeiras ocorrências, na ordem original")


---
## 6. [Prática 4] `most_confident_class`

Depois de um `softmax`, como escolher a classe prevista? Implemente `most_confident_class(scores: list[float]) -> int`: retorna o **índice** do maior valor da lista (o clássico `argmax`).

**Decisão de design pra documentar:** o que fazer em caso de empate entre dois ou mais valores máximos? Não existe resposta errada - o que importa é decidir e testar.

Escopo deliberadamente menor que as outras práticas: depois de implementar com TDD, escrevam só **1 propriedade** com `hypothesis`: o valor no índice retornado é sempre o máximo da lista.

In [ ]:
%%ipytest

import pytest
from hypothesis import given, strategies as st

# TODO: implemente most_confident_class(scores) seguindo TDD.
# Documente aqui a decisão de design para o caso de empate:


def test_most_confident_class_returns_index_of_max():
    pytest.fail("TODO: most_confident_class([0.1, 0.7, 0.2]) -> 1")


def test_most_confident_class_handles_tie():
    # TODO: decida e teste o comportamento em caso de empate
    pytest.fail("TODO: testar most_confident_class([0.5, 0.5, 0.1])")


# TODO: descomente e complete a propriedade abaixo

# @given(st.lists(st.floats(allow_nan=False, allow_infinity=False, width=32), min_size=1))
# def test_most_confident_class_points_to_max_value(scores):
#     pytest.fail("TODO: scores[most_confident_class(scores)] == max(scores)")


---
## 7. [Prática 5] `flag_outliers`

Esse exige raciocínio genuíno, sem resposta óbvia de cara.

Implemente:
```
flag_outliers(scores: list[float], threshold: float = 2.0) -> list[bool]
```
que marca `True` os valores cujo z-score absoluto (`|x - média| / desvio_padrão`) ultrapassa `threshold`.

**Edge case pra resolver:** lista com desvio-padrão zero (todos os valores iguais) - não dá pra dividir por zero, e logicamente não deveria haver outlier nesse caso.

**Propriedade metamórfica** (lembram do teste metamórfico mencionado na Aula 1?): em vez de verificar uma entrada isolada, verificamos uma **relação entre duas entradas relacionadas**. Peguem qualquer lista, insiram um valor absurdamente fora da escala (ex: 1000x o maior valor da lista), e a função precisa **sempre** marcar esse valor inserido como outlier - não importa qual era a lista original.

In [ ]:
%%ipytest

import statistics
import pytest
from hypothesis import given, strategies as st

# TODO: implemente flag_outliers(scores, threshold=2.0) seguindo TDD.
# Documente aqui a decisão de design para desvio-padrão zero:


def test_flag_outliers_flags_extreme_value():
    pytest.fail("TODO: flag_outliers([1, 2, 2, 1, 2, 1, 2, 100]) deve marcar o 100 como True")


def test_flag_outliers_handles_constant_list():
    pytest.fail("TODO: flag_outliers([5, 5, 5, 5]) não deve ter outliers (std=0)")


# TODO: descomente e complete a propriedade metamórfica abaixo
# ATENÇÃO: min_size=6 é obrigatório aqui, não é escolha arbitrária. O z-score máximo
# matematicamente possível de qualquer ponto numa lista de N valores é sqrt(N-1)
# (desvio-padrão populacional) ou (N-1)/sqrt(N) (desvio-padrão amostral, usado por
# statistics.stdev). Com min_size=3 (lista final de N=4 pontos), o teto é ~1.73 (pop)
# ou ~1.5 (amostral) - sempre abaixo do threshold=2.0, então a propriedade seria
# impossível de satisfazer não importa a implementação. Com min_size=6 (N=7+), o teto
# sobe pra ~2.45 (pop) ou ~2.27 (amostral) - passa de 2.0 com folga nas duas convenções.

# @given(
#     st.lists(
#         st.floats(min_value=-1e3, max_value=1e3, allow_nan=False, allow_infinity=False),
#         min_size=6,
#         max_size=20,
#     )
# )
# def test_flag_outliers_metamorphic_extreme_insertion(scores):
#     valor_absurdo = max(abs(s) for s in scores) * 1000 + 1e6
#     lista_com_outlier = scores + [valor_absurdo]
#     resultado = flag_outliers(lista_com_outlier)
#     pytest.fail("TODO: assert resultado[-1] - o valor inserido precisa ser sempre marcado")


---
## 8. [Bônus opcional] `moving_average`

Pra quem terminar o `flag_outliers` antes do tempo. Sem resposta cobrada no quiz ou no Trabalho 1 - é só pra manter o ritmo de quem for mais rápido.

Implemente `moving_average(scores: list[float], window: int) -> list[float]`, útil pra suavizar métricas de monitoramento de modelo em produção (ex: acurácia medida a cada hora, com ruído).

Se quiser um desafio de propriedade: o que vocês esperariam ser sempre verdade sobre a variância da saída comparada com a variância da entrada?

In [ ]:
%%ipytest

import pytest

# TODO (bônus, opcional): implemente moving_average(scores, window)


def test_moving_average_bonus():
    pytest.fail("TODO opcional: defina o comportamento e teste")
